In [ ]:
# === 09_havuz: 1710 kurumsal mesajı temizle, kuralla etiketle, kuralı doğrula ===
import re
import numpy as np
import pandas as pd

D = "../data/external/"

# --- 1. Oku: bütün sütunlar metin ---
havuz = pd.read_csv(D + "salih_kurumsal_havuz.csv", dtype=str, keep_default_na=False)
n0 = len(havuz)

# --- 2. Ek maskeleme: dün 60 mesajda doğrulanan kuralların hepsi tek fonksiyonda ---
GENEL = r"(?:Müşteri|Musteri|Abone|Yolcu|Üye|Uye|Kullanıcı|Kullanici|Hasta|Misafir|Mükellef|Mukellef)"
KURALLAR = [
    (r"\s*B\d{3}\s*$", ""),                                        # sondaki gönderici kodu (B001...)
    (r"Sal[İIiı]h\s+Can\s+T[ÜUüu]rk", "[İSİM]"),                  # kendi adın, her yazımla
    (r"OPTIMUM HAYAT SAGLIK[^,]*?STI\.*", "[İSİM]"),              # abonelik sahibi şirket
    (r"\b\d{1,2}\s?[A-Z]{1,3}\s?\d{2,4}\b", "[PLAKA]"),           # araç plakası
    (r"\b\d{4}(\s+ile biten)", r"0000\1"),                        # kart son 4 hane
    (r"\b(?:G-)?\d{3}-?\d{3}\b(?=\s*(?:doğrulama|dogrulama|is your|IsCep|\.?$))", "[KOD]"),
    (r"^\d{5}(\s+doğrulama)", r"[KOD]\1"),                        # 5 haneli kod başta
    (r"(kodunuz:?\s*)\d{3}-\d{3}", r"\1[KOD]"),
    (r"(code is:?\s*)\d{6}", r"\1[KOD]"),
    (r"\d{7,}", "0000000"),                                       # uzun numaralar
    (r"\b\d{5,6}(\s+tekil)", r"0000000\1"),
]
HITAP = re.compile(
    rf"\b(Sayın|Sayin|Sn|Sevgili|Merhaba)[\s.,]+(?!{GENEL})"
    r"([A-ZÇĞİÖŞÜ][a-zçğıöşü]+|[A-ZÇĞİÖŞÜ]{2,})"
    r"(?:\s+(?:[A-ZÇĞİÖŞÜ][a-zçğıöşü]+|[A-ZÇĞİÖŞÜ]{2,})){0,2}"
)

def temizle_havuz(metin):
    for desen, yerine in KURALLAR:
        metin = re.sub(desen, yerine, metin, flags=re.IGNORECASE if "Sal" in desen else 0)
    return HITAP.sub(lambda m: f"{m.group(1)} [İSİM]", metin)

havuz["mesaj"] = havuz["mesaj"].apply(temizle_havuz)

# --- 3. Senin 60 örnek mesajın zaten havuzdan seçilmişti: tekrar saymamak için çıkar ---
sal = pd.read_csv(D + "salih_kurumsal_etiketli.csv", dtype=str, keep_default_na=False)
havuz = havuz[~havuz["mesaj"].isin(sal["mesaj"])]

# --- 4. Temizlik sonrası tekrarları at (B kodu silinince bazı mesajlar aynı oldu) ---
havuz = havuz.drop_duplicates("mesaj").reset_index(drop=True)
havuz["grup"] = havuz["gonderen"].str.upper().str.strip()

# --- 5. Kontroller ---
print(f"{n0} → {len(havuz)} mesaj, {havuz['grup'].nunique()} gönderici grubu")
print("Adın kalan:", havuz["mesaj"].str.contains(r"Sal[İIiı]h|T[ÜUüu]rk\b", case=False).sum())
print("Sonda B kodu kalan:", havuz["mesaj"].str.contains(r"B\d{3}$").sum())

In [ ]:
# --- Kalanları incele: sadece eşleşen yerin çevresi (30 karakter) ---
def cevre(metin, desen, n=30):
    m = re.search(desen, metin, re.IGNORECASE)
    return metin[max(0, m.start() - n): m.end() + n]

AD = r"Sal[İIiı]h|T[ÜUüu]rk\b"
print("--- Ad kontrolüne takılanlar ---")
for m in havuz.loc[havuz["mesaj"].str.contains(AD, case=False), "mesaj"].head(24):
    print(" ", cevre(m, AD))

print("\n--- B kodu kalanlar ---")
for m in havuz.loc[havuz["mesaj"].str.contains(r"B\d{3}$"), "mesaj"]:
    print(" ", m[-60:])                                  # mesajın son 60 karakteri

In [ ]:
# --- Ek maskeleme: kontrolde yakalanan yeni kalıplar ---
EK = [
    (r"(?:\s*B\d{3})+\s*$", "", 0),                                      # sonda üst üste B kodları
    (r"Sal[İIiı]h(?:\s+Can)?(?:\s+T[ÜUüu]rk)?", "[İSİM]", re.I),       # sadece ad, ad+orta ad, tam ad
    (r"\b[A-ZÇĞİÖŞÜ]{2,}\s+T[ÜU]RK\b", "[İSİM]", 0),                    # BÜYÜK HARFLE "AD TÜRK" (başka kişi)
    (r"\b(Sayın|Sayin|Sn)[\s.,]+T[ÜUüu]rk\b", r"\1 [İSİM]", re.I),     # "SN. TURK" (soyadla hitap)
    (r"(son\s*4\s*hanes[iİı]\s+)\d{4}", r"\g<1>0000", re.I),           # "SON 4 HANESI 1958 OLAN"
]
for desen, yerine, bayrak in EK:
    havuz["mesaj"] = havuz["mesaj"].str.replace(desen, yerine, regex=True, flags=bayrak)

havuz = havuz.drop_duplicates("mesaj").reset_index(drop=True)

# --- Kontroller: yanlış alarmları (Atatürk, Türk Telekom...) dışarıda bırakan daha net desenler ---
print(len(havuz), "mesaj")
print("Salih kalan:", havuz["mesaj"].str.contains(r"Sal[İIiı]h", case=False).sum())
print("'AD TÜRK' kalan:", havuz["mesaj"].str.contains(r"\b[A-ZÇĞİÖŞÜ]{2,}\s+T[ÜU]RK\b").sum())
print("'SN TURK' kalan:", havuz["mesaj"].str.contains(r"\bS(ayın|ayin|n)[\s.,]+T[ÜU]RK\b", case=False).sum())
print("B kodu kalan:", havuz["mesaj"].str.contains(r"B\d{3}$").sum())

In [ ]:
# --- 1. Kural: mesajda "SMS ret/iptal" seçeneği varsa → reklam, yoksa → meşru ---
RET = re.compile(
    r"sms\s*(?:iptal|ipt|ret|almak\s+istemiyor|almamak)"   # "SMS iptal için", "Sms ipt:", "SMS almak istemiyorsanız"
    r"|\bRET\b|IZINYOK|\bcks\s*:"                           # "RET yazın", "IZINYOK yaz", "Cks: N11 IPTAL"
    r"|yaz(?:ıp|ip|ın|in)?\s*\d{4}"                         # "LCW yazıp 3525", "AR54 yaz 4609"
    r"|\d{4}\s*['’]?\s*(?:e|a|ye|ya)\s+(?:gönder|gonder|ilet)",  # "3525'e gönder", "3342'ye iletin"
    re.IGNORECASE,
)
def kural(metin):
    return "reklam" if RET.search(metin) else "meşru"

# --- 2. Doğrulama: insan etiketli iki set (dolandırıcılık hariç: kural onu ayırt etmeye çalışmıyor) ---
zor = pd.read_csv(D + "zor_gercek_hepsi.csv")
zor = zor[zor["etiket"].isin(["meşru", "reklam"])]
sal = sal[sal["belirsiz"] != "e"]

for ad, df in [("Esra zor set (insan)", zor), ("Salih 58 (Claude+Salih)", sal)]:
    tahmin = df["mesaj"].apply(kural)
    tablo = pd.crosstab(df["etiket"], tahmin, rownames=["gerçek"], colnames=["kural"])
    dogru = (tahmin == df["etiket"]).mean()
    print(f"\n=== {ad}: {len(df)} mesaj, doğruluk {dogru:.1%} ===")
    print(tablo)

# --- 3. Kuralın yanıldığı mesajlar (Esra'nın setinde): neyi kaçırıyor? ---
yanlis = zor[zor["mesaj"].apply(kural) != zor["etiket"]]
print(f"\n--- Kural yanlış: {len(yanlis)} mesaj ---")
for _, s in yanlis.iterrows():
    print(f"  gerçek={s['etiket']:6s} | {s['gonderen'][:12]:12s} | {s['mesaj'][:80]}")

In [ ]:
# --- 1. Kademeli kural: ret → reklam | reklam kelimesi → çekimser | ikisi de yok → meşru ---
NO = r"(?:\d{4}|\[TEL\])"                      # kısa numara ya da maskelenmiş numara
RET = re.compile(
    r"sms\s*(?:iptal|ipt|ret|almak\s+istemiyor|almamak)|\bRET\b|IZINYOK|IPTALIZIN|\bcks\s*:"
    r"|yaz(?:ıp|ip|ın|in|arak)?\s*" + NO +             # "LCW yazıp 3525", "IPTAL yazıp [TEL]"
    r"|" + NO + r"\s*['’]?\s*(?:e|a|ye|ya)\s+(?:gönder|gonder|ilet)"
    r"|(?:iptal|ret)\w*\s+yaz", re.IGNORECASE)
PROMO = re.compile(
    r"kampanya|indirim|f[iı]rsat|hediye|bonus|freespin|freebet|kazan|ba[sş]vur|hemen|ka[cç][iı]rma"
    r"|tl\s*'?den|ba[sş]layan|%\s*\d|\d+\s*%|özel|ozel|yayında|yayinda|davet|eğitim|egitim"
    r"|kontenjan|ücretsiz|ucretsiz", re.IGNORECASE)

def kural3(metin):
    if RET.search(metin):   return "reklam"
    if PROMO.search(metin): return "çekimser"
    return "meşru"

# --- 2. Doğrulama: Esra'nın insan etiketli seti ---
print(pd.crosstab(zor["etiket"], zor["mesaj"].apply(kural3), rownames=["gerçek"], colnames=["kural"]))

# --- 3. Havuzu etiketle, çekimserleri at ---
havuz["etiket"] = havuz["mesaj"].apply(kural3)
print("\nHavuz:", havuz["etiket"].value_counts().to_dict())
etiketli = havuz[havuz["etiket"] != "çekimser"].copy()

# --- 4. Bir göndericinin baskın olmasını engelle: göndericide en fazla 20 mesaj ---
# Önce bütün satırları karıştır, sonra her göndericinin ilk 20'sini al = rastgele 20
etiketli = (etiketli.sample(frac=1, random_state=42)
                    .groupby("grup").head(20)
                    .reset_index(drop=True))
print("Son havuz:", len(etiketli), "mesaj,", etiketli["etiket"].value_counts().to_dict(),
      ",", etiketli["grup"].nunique(), "gönderici")
# --- 5. Kaydet (yalnızca senin bilgisayarında, data/external gitignore'da) ---
etiketli[["mesaj", "etiket", "gonderen", "grup"]].to_csv(
    D + "salih_havuz_kural.csv", index=False, encoding="utf-8-sig")

In [ ]:
# --- Kural etiketlerinin havuzda kontrolü: rastgele 30 mesaj ---
kontrol = etiketli.sample(30, random_state=1)[["etiket", "gonderen", "mesaj"]].reset_index(drop=True)
for i, s in kontrol.iterrows():
    print(f"{i:2d} [{s['etiket']:6s}] {s['gonderen'][:12]:12s} | {s['mesaj'][:110]}")

In [ ]:
# --- 1. Boş / anlamsız mesajları at: resim yer tutucusu (￼) ve 15 karakterden kısa olanlar ---
etiketli["mesaj"] = etiketli["mesaj"].str.replace("\ufffc", "", regex=False).str.strip()
n = len(etiketli)
etiketli = etiketli[etiketli["mesaj"].str.len() >= 15]
print(f"Boş/kısa atıldı: {n - len(etiketli)}")

# --- 2. Başta 6 haneli tek kullanımlık şifre: "759695 Tek kullanimlik" ---
etiketli["mesaj"] = etiketli["mesaj"].str.replace(r"^\d{6}(?=\s)", "[KOD]", regex=True)

# --- 3. Tekrar kaydet ---
etiketli[["mesaj", "etiket", "gonderen", "grup"]].to_csv(
    D + "salih_havuz_kural.csv", index=False, encoding="utf-8-sig")
print(len(etiketli), "mesaj,", etiketli["etiket"].value_counts().to_dict())

In [ ]:
# --- Şablon bazında tekrar: rakamlar sıfırlanınca aynı olanlar tek sayılır ---
kural_etiketli = havuz[havuz["etiket"] != "çekimser"].copy()
kural_etiketli["sablon"] = kural_etiketli["mesaj"].str.replace(r"\d", "0", regex=True)

sablonsuz = kural_etiketli.drop_duplicates("sablon")
print(f"Kural etiketli: {len(kural_etiketli)}  →  farklı şablon: {len(sablonsuz)}")
print("En kalabalık 5 gönderici (şablon sonrası):")
print(sablonsuz["grup"].value_counts().head())

In [ ]:
# --- 1. Kural etiketli + şablon sütunu ---
etiketli = havuz[havuz["etiket"] != "çekimser"].copy()
etiketli["sablon"] = etiketli["mesaj"].str.replace(r"\d", "0", regex=True)

# --- 2. Şablon tekrarını at (rakamları farklı aynı mesajlar tek sayılır) ---
etiketli = etiketli.drop_duplicates("sablon")

# --- 3. Göndericide en fazla 40 (karıştır → her göndericinin ilk 40'ı) ---
etiketli = (etiketli.sample(frac=1, random_state=42)
                    .groupby("grup").head(40)
                    .reset_index(drop=True))

# --- 4. Temizlik: boş/kısa mesajlar, baştaki 6 haneli kod ---
etiketli["mesaj"] = etiketli["mesaj"].str.replace("\ufffc", "", regex=False).str.strip()
etiketli = etiketli[etiketli["mesaj"].str.len() >= 15]
etiketli["mesaj"] = etiketli["mesaj"].str.replace(r"^\d{6}(?=\s)", "[KOD]", regex=True)

# --- 5. Kaydet ---
etiketli[["mesaj", "etiket", "gonderen", "grup"]].to_csv(
    D + "salih_havuz_kural.csv", index=False, encoding="utf-8-sig")
print(len(etiketli), "mesaj,", etiketli["etiket"].value_counts().to_dict(),
      ",", etiketli["grup"].nunique(), "gönderici")
print("En kalabalık:", etiketli["grup"].value_counts().head(3).to_dict())

In [ ]:
# --- Üç havuz versiyonu: hangisi daha iyi, G deneyi söyleyecek ---
def temizle_son(df):
    df = df.copy()
    df["mesaj"] = df["mesaj"].str.replace("\ufffc", "", regex=False).str.strip()
    df = df[df["mesaj"].str.len() >= 15]                                   # boş/kısa mesajlar
    df["mesaj"] = df["mesaj"].str.replace(r"^\d{6}(?=\s)", "[KOD]", regex=True)
    return df

# --- 1. Hepsi: kural etiketli, tekrarlar dahil ---
tum = temizle_son(havuz[havuz["etiket"] != "çekimser"])
tum["sablon"] = tum["mesaj"].str.replace(r"\d", "0", regex=True)

# --- 2. Şablon tekrarı atılmış, sınır yok ---
sablon = tum.drop_duplicates("sablon")

# --- 3. Şablon + göndericide en fazla 40 ---
s40 = sablon.sample(frac=1, random_state=42).groupby("grup").head(40)

# --- 4. Kaydet ve özetle ---
for ad, df in [("tum", tum), ("sablon", sablon), ("40", s40)]:
    df[["mesaj", "etiket", "gonderen", "grup"]].to_csv(
        D + f"salih_havuz_{ad}.csv", index=False, encoding="utf-8-sig")
    print(f"HV_{ad:7s} {len(df):5d} mesaj  {df['etiket'].value_counts().to_dict()}")